In [1]:
import numpy as np
import pandas as pd
import matplotlib as pyplt


In [2]:
import os

data_file = "customer_churn_sample(1).csv"
if not os.path.isfile(data_file):
    data_file = "customer_churn_sample(1).csv"
    if not os.path.isfile(data_file):
        alternative_files = []
        for root, _, files in os.walk("."):
            for file_name in files:
                if file_name.lower().startswith("customer_churn_sample") and file_name.lower().endswith(".csv"):
                    alternative_files.append(os.path.join(root, file_name))

        if alternative_files:
            data_file = alternative_files[0]
        else:
            raise FileNotFoundError(
                f"Could not find {data_file}. "
                "Place the CSV in the notebook folder or update `data_file` with the correct path."
            )

    df = pd.read_csv(data_file)

df = pd.read_csv(data_file)



In [3]:
df.shape

(15, 11)

In [4]:
df.head()

,CustomerID,Gender,Age,TenureMonths,SubscriptionType,MonthlyCharges,TotalCharges,ContractType,SupportTickets,PaymentMethod,Churn
0,CUST-1001,Female,34,12,Basic,49.99,599.88,Month-to-Month,3,Credit Card,Yes
1,CUST-1002,Male,45,24,Pro,79.99,1919.76,One Year,1,Bank Transfer,No
2,CUST-1003,Female,29,6,Basic,49.99,299.94,Month-to-Month,5,UPI,Yes
3,CUST-1004,Male,52,36,Enterprise,149.99,5399.64,Two Year,0,Credit Card,No
4,CUST-1005,Female,41,8,Pro,79.99,639.92,Month-to-Month,4,Debit Card,Yes


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        15 non-null     str    
 1   Gender            15 non-null     str    
 2   Age               15 non-null     int64  
 3   TenureMonths      15 non-null     int64  
 4   SubscriptionType  15 non-null     str    
 5   MonthlyCharges    15 non-null     float64
 6   TotalCharges      15 non-null     float64
 7   ContractType      15 non-null     str    
 8   SupportTickets    15 non-null     int64  
 9   PaymentMethod     15 non-null     str    
 10  Churn             15 non-null     str    
dtypes: float64(2), int64(3), str(6)
memory usage: 2.0 KB


In [5]:
df.describe()

,Age,TenureMonths,MonthlyCharges,TotalCharges,SupportTickets
count,15.000000,15.000000,15.000000,15.000000,15.000000
mean,39.066667,18.800000,84.656667,2088.478667,2.533333
std,11.392144,14.333776,42.739521,2407.365767,1.922300
min,23.000000,2.000000,49.990000,99.980000,0.000000
25%,30.000000,7.000000,49.990000,449.910000,1.000000
50%,38.000000,15.000000,79.990000,1099.780000,2.000000
75%,47.000000,27.000000,114.990000,3209.730000,4.000000
max,60.000000,48.000000,149.990000,7199.520000,6.000000


In [6]:
def to_snake_case(column_name: str) -> str:
    """Convert column headings to snake_case."""
    column_name = str(column_name).strip()
    column_name = re.sub(
        r"(.)([A-Z][a-z]+)",
        r"\1_\2",
        column_name,
    )
    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )
    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )
    return column_name.strip("_").lower()


In [19]:
from abc import ABC, abstractmethod

class column(ABC):
    def __init__(self, name: str, values):
        self.name = str(name)
        self.values = pd.Series(values)

    @abstractmethod
    def clean(self) -> pd.Series:
        raise NotImplementedError

    @abstractmethod
    def validate(self) -> bool:
        raise NotImplementedError

    @abstractmethod
    def summary(self) -> dict:
        raise NotImplementedError

class DataColumn(column):
    def clean(self) -> pd.Series:
        cleaned = self.values.copy()

        if cleaned.dtype == object:
            cleaned = cleaned.astype("string").str.strip()

        cleaned = cleaned.replace(
            [
                "",
                " ",
                "NA",
                "N/A",
                "na",
                "n/a",
                "NULL",
                "null",
                "None",
                "none",
                "-",
                "?",
            ],
            np.nan,
        )

        if pd.api.types.is_numeric_dtype(cleaned):
            cleaned = pd.to_numeric(cleaned, errors="coerce")

        self.values = cleaned
        return self.values

    def validate(self) -> bool:
        return bool(self.values.notna().all())

    def summary(self) -> dict:
        return {
            "name": self.name,
            "dtype": str(self.values.dtype),
            "count": int(self.values.count()),
            "missing": int(self.values.isna().sum()),
            "unique": int(self.values.nunique(dropna=True)),
        }

In [20]:
text_columns = df.select_dtypes(
        include="object"
    ).columns

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
    )

C:\Users\soodr\AppData\Local\Temp\ipykernel_21928\2636456428.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(


In [21]:
import numpy as np
import pandas as pd
import re

null_values = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none",
    "-",
    "?",
]

df = df.replace(null_values, np.nan)
df.columns = [to_snake_case(col) for col in df.columns]

missing_before = (
    df.isna()
    .sum()
    .astype(int)
    .to_dict()
)

df = df.replace(null_values, np.nan)
df.columns = [to_snake_case(col) for col in df.columns]

missing_before = (
    df.isna()
    .sum()
    .astype(int)
    .to_dict()
)

df = df.replace(null_values, np.nan)

missing_before = (
    df.isna()
    .sum()
    .astype(int)
    .to_dict()
)

missing_before = (
    df.isna()
    .sum()
    .astype(int)
    .to_dict()
)

In [22]:
category_mappings = {
    "gender": {
        "female": "Female",
        "male": "Male"
    },
    "subscription_type": {
        "basic": "Basic",
        "pro": "Pro",
        "enterprise": "Enterprise"
    },
    "contract_type": {
        "month-to-month": "Month-to-Month",
        "month to month": "Month-to-Month",
        "one year": "One Year",
        "two year": "Two Year"
    },
    "payment_method": {
        "credit card": "Credit Card",
        "debit card": "Debit Card",
        "bank transfer": "Bank Transfer",
        "upi": "UPI"
    },
    "churn": {
        "yes": "Yes",
        "no": "No"
    }
}

for column, mapping in category_mappings.items():
    if column in df.columns:
        normalized_values = (
            df[column]
            .astype("string")
            .str.lower()
            .str.strip()
        )

        df[column] = (
            normalized_values
            .map(mapping)
            .fillna(df[column])
        )

In [23]:
integer_columns = [
    "age",
    "tenure_months",
    "support_tickets",
]

decimal_columns = [
    "monthly_charges",
    "total_charges",
]

for column in integer_columns + decimal_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

In [24]:
rows_before_id_drop = len(df)

if "customer_id" in df.columns:
        df = df.dropna(
            subset=["customer_id"]
        )

rows_dropped_missing_id = (
        rows_before_id_drop - len(df)
    )

numeric_imputation = {}

for column in integer_columns + decimal_columns:
        if column in df.columns:
            missing_count = int(
                df[column].isna().sum()
            )

            if missing_count > 0:
                median_value = df[column].median()

                df[column] = df[column].fillna(
                    median_value
                )

                numeric_imputation[column] = {
                    "filled": missing_count,
                    "median": float(median_value),
                }
            else:
                numeric_imputation[column] = {
                    "filled": 0,
                    "median": None,
                }

categorical_columns = [
        "gender",
        "subscription_type",
        "contract_type",
        "payment_method",
        "churn",
    ]

categorical_imputation = {}

for column in categorical_columns:
        if column in df.columns:
            missing_count = int(
                df[column].isna().sum()
            )

            df[column] = df[column].fillna(
                "Unknown"
            )

            categorical_imputation[column] = {
                "filled": missing_count,
                "replacement": (
                    "Unknown"
                    if missing_count
                    else None
                ),
            }

In [25]:
for column in integer_columns:
        if column in df.columns:
            df[column] = (
                df[column]
                .round()
                .astype("Int64")
            )

In [26]:
duplicate_rows = int(
        df.duplicated().sum()
    )

df = (
        df.drop_duplicates()
        .reset_index(drop=True)
    )

duplicate_customer_ids = []

if "customer_id" in df.columns:
        duplicate_customer_ids = (
            df.loc[
                df["customer_id"].duplicated(
                    keep=False
                ),
                "customer_id",
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

In [29]:
charge_mismatch_count = 0
charge_mismatch_rows = []

required_columns = {
        "monthly_charges",
        "tenure_months",
        "total_charges",
    }
if required_columns.issubset(df.columns):
        expected_total = (
            df["monthly_charges"]
            * df["tenure_months"].astype(float)
        ).round(2)

        mismatch_mask = ~np.isclose(
            expected_total.to_numpy(dtype=float),
            df["total_charges"].to_numpy(
                dtype=float
            ),
            atol=0.01,
            equal_nan=True,
        )

        charge_mismatch_count = int(
            mismatch_mask.sum()
        )

        if charge_mismatch_count > 0:
            columns_to_show = [
                column
                for column in [
                    "customer_id",
                    "monthly_charges",
                    "tenure_months",
                    "total_charges",
                ]
                if column in df.columns
            ]

            mismatch_data = df.loc[
                mismatch_mask,
                columns_to_show,
            ].copy()

            mismatch_data[
                "expected_total_charges"
            ] = expected_total[mismatch_mask]

            charge_mismatch_rows = (
                mismatch_data.to_dict(
                    orient="records"
                )
         )

In [28]:

df.to_csv("cleaned_dataset.csv", index=False)